## Model Problem A: Gaussian Targets

## Model Problem B: A Simple Statistical Inversion in 2 Dimensions

We consider the statistical inversion problem of estimating $(x,y) \in \mathbb{R}^2$ from data $z \in \mathbb{R}$ gathered according the measurement model:
$$
    z = f(x,y) + \eta \qquad \text{ where } \eta \sim N(0, \sigma^2) \text{ and } f: \mathbb{R}^2 \to \mathbb{R}
    \text{ is the forward function. }
    \tag{MM1}
$$
Leaving aside the measurement noise $\eta$ we would like to recover $(x,y) = f^{-1}(z)$.  This typically be impossible in priciple.  However, if we put a Gaussian prior $\mu_0 \sim N(0, C)$ on $(x,y)$ then the posterior 'bayesian solution' $(x,y)|z$ is the distribution:
$$
    \mu(dx, dy) \propto \exp\left( - \frac{1}{2 \sigma^2} ( f(x,y) - z)^2 
    - \frac{1}{2} <C^{-1}(x,y), (x,y)>\right) dx dy.
    \tag{S1}
$$

## Model Problem C: Matrix Coefficents from Solutions

We next consider inverse problem of estimating the coefficents of an $n \times n$ anti-semetric matrix $A$ from data $\theta \in \mathbb{R}^n$ gathered according the measurement model:
$$
    z = \mathcal{O}(\theta(A)) + \eta \qquad \text{ where } \eta \sim N(0, \sigma^2)
    \tag{MM2}
$$
Here $\theta := \theta(A)$ is the solution of
$$
    (A + \kappa I)\theta = g
$$
and where $\mathcal{O}(\theta)$ defined by $\mathcal{O}: \mathbb{R}^n \to \mathbb{R}^m$ is a partial observation of $\theta$. Typically
$$
    \mathcal{O}((\theta_1, \ldots, \theta_n)) = (\theta_1,\ldots, \theta_m)
$$
Here the Bayesian posterior will take the form
$$
    \mu(dA) \propto \exp\left( - \frac{1}{2 \sigma^2} 
    | \mathcal{O}( (A +\kappa I)^{-1} g) - z|^2 
    - \frac{1}{2} <C^{-1}A, A>\right) dA.
    \tag{S1}
$$
which is a target in $d = n(n-1)/2$






In [1]:
#Set-up: Packages to Import and Untility Functions

#Core Numerical Packages for Paral
import MCMC_Sampliers as MCMCsmp

#Numerical Elements
from numpy.linalg import norm
import numpy as np
from numpy import dot, array, transpose, diag
import random
import math

#Input Output utils
import os
import pandas as pd

#Stats elements
from scipy.stats import norm

#Plotting stuff
from matplotlib.patches import Ellipse

#Finished Message
import smtplib
from email.message import EmailMessage

from functools import partial
import multiprocessing as mp

print("Current cpu count:" + str(mp.cpu_count()))

Current cpu count:24


In [2]:
#Experiment 1

#Gaussian target (changing Covariance/mean on `low modes')

#Model dim
TarDimEx1 = 5

#Covariance algebraic decay
cov0 = 4
gam = 2
CovDiag_p = [cov0* (j**(-gam)) for j in list(range(1,TarDimEx1+1))]

CovPriorEx1 = MCMCsmp.mkDiagCov(CovDiag_p)
print("Prior Covariance:")
print(CovPriorEx1)

#Information for Gaussian Posterior
#Perturb dim
pertDimEx1 = 2
CovDiag_post = [CovDiag_p[1], CovDiag_p[0]] + CovDiag_p[pertDimEx1:]
CovPostEx1 = MCMCsmp.mkDiagCov(CovDiag_post)
meanPostEx1 = np.zeros(TarDimEx1)

print("Posterior Covariance:")
print(CovPostEx1)


#Make Problem Information File
expId =  2
FileNmBase = "Data/Large_p_study/Experiment_1/" + "Ex_ID_"+ str(expId) + "/"
os.makedirs(FileNmBase, exist_ok=True)

probDataFile = FileNmBase + "Problem_Info.txt"
with open(probDataFile, 'w') as file:
    file.write("Target Type:  Model Problem A -- Gaussian Target \n")
    file.write("Target Dimension: " + str(TarDimEx1) + "\n")
    file.write("Prior Covariance: \n" + str(CovPriorEx1) + "\n")
    file.write("Perturbation Dimension: " + str(pertDimEx1) + "\n")
    file.write("Posterior Mean: " + str(meanPostEx1) + "\n")
    file.write("Posterior Covariance: \n" + str(CovPostEx1)+ "\n")

PriorCovInvEx1 = MCMCsmp.mkDiagCov([CovDiag_p[0]**(-1),CovDiag_p[1]**(-1)])
PostCovInvEx1 = MCMCsmp.mkDiagCov([CovDiag_post[0]**(-1),CovDiag_post[1]**(-1)])

PotEx1Pass = partial(MCMCsmp.PotGaussPert, TarDim=TarDimEx1, PertDim=pertDimEx1, PriorCovInv=PriorCovInvEx1, PostMean=meanPostEx1[0:pertDimEx1], PostCovInv=PostCovInvEx1, mode = "soft")


ImpList = [[5,20,20000,20,1000], [10,20,20000,20,1000]]
#FORMAT: [p,NumRho,NumSampsESS,numChainsESS, NumChainsgM, q0zSt]
#            -p: value to p to run study
#            -NumRho: 1/NumRho specfies the step size in rho over [0,1] for the study
#            -NumSampsESS: Length of the MCMC at each rho value to compute ESS/MSJD
#            -numChainsESS: Number of independent chains to compute ESS/MSJD
#            -NumChainsgM: Number of separate chain M= NumChainsgM to compute Var(\bar{g}_N^\rho)





Prior Covariance:
[[4.         0.         0.         0.         0.        ]
 [0.         1.         0.         0.         0.        ]
 [0.         0.         0.44444444 0.         0.        ]
 [0.         0.         0.         0.25       0.        ]
 [0.         0.         0.         0.         0.16      ]]
Posterior Covariance:
[[1.         0.         0.         0.         0.        ]
 [0.         4.         0.         0.         0.        ]
 [0.         0.         0.44444444 0.         0.        ]
 [0.         0.         0.         0.25       0.        ]
 [0.         0.         0.         0.         0.16      ]]


In [3]:

q0FN = lambda nmICs: np.random.multivariate_normal(np.zeros(TarDimEx1), CovPostEx1, size=nmICs)

if __name__ == "__main__":
    par_sweep_dictEx1 = MCMCsmp.parameter_sweep_p_rho(ImpList, q0FN, TarDimEx1, CovPriorEx1, PotEx1Pass)

MCMCsmp.parameter_sweep_p_rho_save_figures(par_sweep_dictEx1 , TarDimEx1, FileNmBase)

MCMCsmp.message_me("Round 1 Complete","p = 5, 10")


ImpList = [[20,20,20000,20,1000], [40,20,20000,20,1000], [80,20,20000,20,1000],[160,20,20000,20,1000],[320,20,20000,20,1000]]


q0FN = lambda nmICs: np.random.multivariate_normal(np.zeros(TarDimEx1), CovPostEx1, size=nmICs)

if __name__ == "__main__":
    par_sweep_dictEx1 = MCMCsmp.parameter_sweep_p_rho(ImpList, q0FN, TarDimEx1, CovPriorEx1, PotEx1Pass)

MCMCsmp.parameter_sweep_p_rho_save_figures(par_sweep_dictEx1 , TarDimEx1, FileNmBase)

MCMCsmp.message_me("Round 1 Complete","p = 20, 40, 80, 160, 320")


Currently running: p=5
Delta rho: 0.05
Number of Samples per Chain to compute ESS/MSJD: 20000
Number of Independent Chain to compute ESS/MSJD: 20
N in bar{g}_N^rho = 1/N sum_{j =1}^N g(X_jrho): 100
Number of separate chain M to compute Var(bar{g}_Nrho): 1000
Total MCMC Runs Submitted: 57


Parallel MCMC Runs:   0%|          | 0/57 [00:00<?, ?it/s]

Currently running: p=10
Delta rho: 0.05
Number of Samples per Chain to compute ESS/MSJD: 20000
Number of Independent Chain to compute ESS/MSJD: 20
N in bar{g}_N^rho = 1/N sum_{j =1}^N g(X_jrho): 100
Number of separate chain M to compute Var(bar{g}_Nrho): 1000
Total MCMC Runs Submitted: 114


Parallel MCMC Runs:   0%|          | 0/114 [00:00<?, ?it/s]

Building ESS and MSJD Graphics:   0%|          | 0/2 [00:00<?, ?it/s]

...

Currently running: p=160
Delta rho: 0.05
Number of Samples per Chain to compute ESS/MSJD: 20000
Number of Independent Chain to compute ESS/MSJD: 20
N in bar{g}_N^rho = 1/N sum_{j =1}^N g(X_jrho): 100
Number of separate chain M to compute Var(bar{g}_Nrho): 1000
Total MCMC Runs Submitted: 228


Parallel MCMC Runs:   0%|          | 0/228 [00:00<?, ?it/s]

Currently running: p=320
Delta rho: 0.05
Number of Samples per Chain to compute ESS/MSJD: 20000
Number of Independent Chain to compute ESS/MSJD: 20
N in bar{g}_N^rho = 1/N sum_{j =1}^N g(X_jrho): 100
Number of separate chain M to compute Var(bar{g}_Nrho): 1000
Total MCMC Runs Submitted: 285


Parallel MCMC Runs:   0%|          | 0/285 [00:00<?, ?it/s]

Building ESS and MSJD Graphics:   0%|          | 0/5 [00:00<?, ?it/s]

In [6]:
import time

#Experiment 1

#Gaussian target (changing Covariance/mean on `low modes')


#Model dim
TarDimEx1 = 20

#Covariance algebraic decay
cov0 = 4
gam = 2
CovDiag_p = [cov0* (j**(-gam)) for j in list(range(1,TarDimEx1+1))]

CovPriorEx1 = MCMCsmp.mkDiagCov(CovDiag_p)
#print("Prior Covariance:")
#print(CovPriorEx1)

#Information for Gaussian Posterior
#Perturb dim
pertDimEx1 = 2
CovDiag_post = [CovDiag_p[1], CovDiag_p[0]] + CovDiag_p[pertDimEx1:]
CovPostEx1 = MCMCsmp.mkDiagCov(CovDiag_post)
meanPostEx1 = np.zeros(TarDimEx1)

#print("Posterior Covariance:")
#print(CovPostEx1)


#Make Problem Information File
expId =  3
FileNmBase = "Data/Large_p_study/Experiment_1/" + "Ex_ID_"+ str(expId) + "/"
os.makedirs(FileNmBase, exist_ok=True)

probDataFile = FileNmBase + "Problem_Info.txt"
with open(probDataFile, 'w') as file:
    file.write("Target Type:  Model Problem A -- Gaussian Target \n")
    file.write("Target Dimension: " + str(TarDimEx1) + "\n")
    file.write("Prior Covariance: \n" + str(CovPriorEx1) + "\n")
    file.write("Perturbation Dimension: " + str(pertDimEx1) + "\n")
    file.write("Posterior Mean: " + str(meanPostEx1) + "\n")
    file.write("Posterior Covariance: \n" + str(CovPostEx1)+ "\n")

PriorCovInvEx1 = MCMCsmp.mkDiagCov([CovDiag_p[0]**(-1),CovDiag_p[1]**(-1)])
PostCovInvEx1 = MCMCsmp.mkDiagCov([CovDiag_post[0]**(-1),CovDiag_post[1]**(-1)])

PotEx1Pass = partial(MCMCsmp.PotGaussPert, TarDim=TarDimEx1, PertDim=pertDimEx1, PriorCovInv=PriorCovInvEx1, PostMean=meanPostEx1[0:pertDimEx1], PostCovInv=PostCovInvEx1, mode = "soft")

start_time = time.perf_counter()

#ImpList = [[5,20,200,20,3], [10,20,200,20,3]]

#FORMAT: [p,NumRho,NumSampsESS,numChainsESS, NumChainsgM, q0zSt]
#            -p: value to p to run study
#            -NumRho: 1/NumRho specfies the step size in rho over [0,1] for the study
#            -NumSampsESS: Length of the MCMC at each rho value to compute ESS/MSJD
#            -numChainsESS: Number of independent chains to compute ESS/MSJD
#            -NumChainsgM: Number of separate chain M= NumChainsgM to compute Var(\bar{g}_N^\rho)

ImpList = [[5,50,20000,100,50000],[10,50,20000,100,50000],[20,50,20000,20,50000],[40,50,20000,100,50000],[80,50,20000,100,50000],[160,50,20000,100,50000],[320,50,20000,100,50000],[640,50,20000,100,50000],[1080,50,20000,100,50000]]


q0FN = lambda nmICs: np.random.multivariate_normal(np.zeros(TarDimEx1), CovPostEx1, size=nmICs)

if __name__ == "__main__":
    par_sweep_dictEx1 = MCMCsmp.parameter_sweep_p_rho(ImpList, q0FN, TarDimEx1, CovPriorEx1, PotEx1Pass)

MCMCsmp.parameter_sweep_p_rho_save_figures(par_sweep_dictEx1 , TarDimEx1, FileNmBase)

# Record the end time
end_time = time.perf_counter()

# Calculate the elapsed time
elapsed_time = end_time - start_time

MCMCsmp.message_me("Round 1 Complete for dim 20","p = 5, 10, 20, 40, 80, 160, 320, 640, 1080..... Total computation time: " + str(elapsed_time))


[[4.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.        ]
 [0.         1.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.        ]
 [0.         0.         0.44444444 0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.        ]
 [0.         0.         0.         0.25       0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.        ]
 [0.         0.         0.         0.         0.16       0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.


Parallel MCMC Runs:   0%|          | 0/57 [00:00<?, ?it/s]

Currently running: p=10
Delta rho: 0.05
Number of Samples per Chain to compute ESS/MSJD: 200
Number of Independent Chain to compute ESS/MSJD: 20
N in bar{g}_N^rho = 1/N sum_{j =1}^N g(X_jrho): 100
Number of separate chain M to compute Var(bar{g}_Nrho): 3
Total MCMC Runs Submitted: 114


Parallel MCMC Runs:   0%|          | 0/114 [00:00<?, ?it/s]

Building ESS and MSJD Graphics:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
import time

#Experiment 1

#Gaussian target (changing Covariance/mean on `low modes')


#Model dim
TarDimEx1 = 40

#Covariance algebraic decay
cov0 = 4
gam = 2
CovDiag_p = [cov0* (j**(-gam)) for j in list(range(1,TarDimEx1+1))]

CovPriorEx1 = MCMCsmp.mkDiagCov(CovDiag_p)
#print("Prior Covariance:")
#print(CovPriorEx1)

#Information for Gaussian Posterior
#Perturb dim
pertDimEx1 = 2
CovDiag_post = [CovDiag_p[1], CovDiag_p[0]] + CovDiag_p[pertDimEx1:]
CovPostEx1 = MCMCsmp.mkDiagCov(CovDiag_post)
meanPostEx1 = np.zeros(TarDimEx1)

#print("Posterior Covariance:")
#print(CovPostEx1)


#Make Problem Information File
expId =  4
FileNmBase = "Data/Large_p_study/Experiment_1/" + "Ex_ID_"+ str(expId) + "/"
os.makedirs(FileNmBase, exist_ok=True)

probDataFile = FileNmBase + "Problem_Info.txt"
with open(probDataFile, 'w') as file:
    file.write("Target Type:  Model Problem A -- Gaussian Target \n")
    file.write("Target Dimension: " + str(TarDimEx1) + "\n")
    file.write("Prior Covariance: \n" + str(CovPriorEx1) + "\n")
    file.write("Perturbation Dimension: " + str(pertDimEx1) + "\n")
    file.write("Posterior Mean: " + str(meanPostEx1) + "\n")
    file.write("Posterior Covariance: \n" + str(CovPostEx1)+ "\n")

PriorCovInvEx1 = MCMCsmp.mkDiagCov([CovDiag_p[0]**(-1),CovDiag_p[1]**(-1)])
PostCovInvEx1 = MCMCsmp.mkDiagCov([CovDiag_post[0]**(-1),CovDiag_post[1]**(-1)])

PotEx1Pass = partial(MCMCsmp.PotGaussPert, TarDim=TarDimEx1, PertDim=pertDimEx1, PriorCovInv=PriorCovInvEx1, PostMean=meanPostEx1[0:pertDimEx1], PostCovInv=PostCovInvEx1, mode = "soft")

start_time = time.perf_counter()

#ImpList = [[5,20,200,20,3], [10,20,200,20,3]]

#FORMAT: [p,NumRho,NumSampsESS,numChainsESS, NumChainsgM, q0zSt]
#            -p: value to p to run study
#            -NumRho: 1/NumRho specfies the step size in rho over [0,1] for the study
#            -NumSampsESS: Length of the MCMC at each rho value to compute ESS/MSJD
#            -numChainsESS: Number of independent chains to compute ESS/MSJD
#            -NumChainsgM: Number of separate chain M= NumChainsgM to compute Var(\bar{g}_N^\rho)

ImpList = [[5,50,20000,100,50000],[10,50,20000,100,50000],[20,50,20000,20,50000],[40,50,20000,100,50000],[80,50,20000,100,50000],[160,50,20000,100,50000],[320,50,20000,100,50000],[640,50,20000,100,50000],[1080,50,20000,100,50000]]


q0FN = lambda nmICs: np.random.multivariate_normal(np.zeros(TarDimEx1), CovPostEx1, size=nmICs)

if __name__ == "__main__":
    par_sweep_dictEx1 = MCMCsmp.parameter_sweep_p_rho(ImpList, q0FN, TarDimEx1, CovPriorEx1, PotEx1Pass)

MCMCsmp.parameter_sweep_p_rho_save_figures(par_sweep_dictEx1 , TarDimEx1, FileNmBase)

# Record the end time
end_time = time.perf_counter()

# Calculate the elapsed time
elapsed_time = end_time - start_time

MCMCsmp.message_me("Round 1 Complete for dim 40","p = 5, 10, 20, 40, 80, 160, 320, 640, 1080..... Total computation time: " + str(elapsed_time))